In [3]:
import numpy as np
import matplotlib.pyplot as plt
import os 



In [6]:
import re

files = []
path = './output/'
for i in os.listdir(path):
    if i.endswith('.dat'):
        files.append(i)

# sort files by numeric timestep in names like at0.dat, at500.dat, ...
def extract_step(filename):
    m = re.search(r'\d+', filename)
    return int(m.group()) if m else -1

files = sorted(files, key=extract_step)
print(f'Found {len(files)} data files')
if not files:
    raise FileNotFoundError(f'No .dat files found in {path}')
print('First file:', files[0], '| Last file:', files[-1])


Found 2001 data files
First file: at0.dat | Last file: at20000.dat


In [41]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, 50)
ax.set_ylim(0, 50)
ax.set_aspect('equal', adjustable='box')
ax.set_title('Vicsek Model Animation')
ax.set_xlabel('x')
ax.set_ylabel('y')

# initialize with first frame
first_file = os.path.join(path, files[0])
data0 = np.genfromtxt(first_file, delimiter=' ', skip_header=1)
scat = ax.scatter(data0[:, 0], data0[:, 1], s=12, c='tab:blue')
label = ax.text(0.02, 0.98, '', transform=ax.transAxes, va='top')

def update(frame_idx):
    file = os.path.join(path, files[frame_idx])
    data = np.genfromtxt(file, delimiter=' ', skip_header=1)
    scat.set_offsets(data[:, :2])
    label.set_text(f'frame: {frame_idx + 1}/{len(files)} | file: {files[frame_idx]}')
    return scat, label

ani = FuncAnimation(fig, update, frames=len(files), interval=120, blit=False, repeat=True)
plt.close(fig)
HTML(ani.to_jshtml())


In [7]:
# Save animation to MP4 (requires ffmpeg installed on your system)
from matplotlib.animation import FuncAnimation, FFMpegWriter

fig_mp4, ax_mp4 = plt.subplots(figsize=(6, 6))
ax_mp4.set_xlim(0, 50)
ax_mp4.set_ylim(0, 50)
ax_mp4.set_aspect('equal', adjustable='box')
ax_mp4.set_title('Vicsek Model Animation (MP4)')
ax_mp4.set_xlabel('x')
ax_mp4.set_ylabel('y')

data0_mp4 = np.genfromtxt(os.path.join(path, files[0]), delimiter=' ', skip_header=1)
scat_mp4 = ax_mp4.scatter(data0_mp4[:, 0], data0_mp4[:, 1], s=12, c='tab:blue')
label_mp4 = ax_mp4.text(0.02, 0.98, '', transform=ax_mp4.transAxes, va='top')

def update_mp4(frame_idx):
    data = np.genfromtxt(os.path.join(path, files[frame_idx]), delimiter=' ', skip_header=1)
    scat_mp4.set_offsets(data[:, :2])
    label_mp4.set_text(f'frame: {frame_idx + 1}/{len(files)} | file: {files[frame_idx]}')
    return scat_mp4, label_mp4

ani_mp4 = FuncAnimation(fig_mp4, update_mp4, frames=len(files), interval=120, blit=False, repeat=False)

output_mp4 = 'vicsek_animation.mp4'
writer = FFMpegWriter(fps=12, bitrate=1800)

try:
    ani_mp4.save(output_mp4, writer=writer, dpi=150)
    print(f'Saved {output_mp4}')
except FileNotFoundError:
    print('ffmpeg not found. Install ffmpeg, then run this cell again.')

plt.close(fig_mp4)


Saved vicsek_animation.mp4


In [ ]:
# MP4: each particle has its own color; trail fades from transparent to full color (start -> end)
from matplotlib.collections import LineCollection
from matplotlib.animation import FuncAnimation, FFMpegWriter
import matplotlib.cm as cm

fps = 12
Lx, Ly = 50.0, 50.0  # periodic box size

fig_trail, ax_trail = plt.subplots(figsize=(6, 6))
ax_trail.set_xlim(0, Lx)
ax_trail.set_ylim(0, Ly)
ax_trail.set_aspect('equal', adjustable='box')
ax_trail.set_title('Vicsek Model - Per-Particle Colored Paths')
ax_trail.set_xlabel('x')
ax_trail.set_ylabel('y')

first = np.genfromtxt(os.path.join(path, files[0]), delimiter=' ', skip_header=1)[:, :2]
n_particles = first.shape[0]

# Assign one distinct color per particle
particle_cmap = cm.get_cmap('tab20' if n_particles <= 20 else 'hsv')
particle_colors = [particle_cmap(i / max(1, n_particles - 1)) for i in range(n_particles)]

prev_wrapped   = first.copy()
prev_unwrapped = first.copy()

segment_histories = [[] for _ in range(n_particles)]

# Scatter: each dot colored by particle index
scat_colors = np.array([particle_colors[i][:3] for i in range(n_particles)])
scat_trail = ax_trail.scatter(first[:, 0], first[:, 1], s=18, c=scat_colors, zorder=3)

# One LineCollection per particle
line_collections = []
for i in range(n_particles):
    lc = LineCollection([], colors=[], linewidths=1.4, zorder=2)
    ax_trail.add_collection(lc)
    line_collections.append(lc)

label_trail = ax_trail.text(0.02, 0.98, '', transform=ax_trail.transAxes, va='top')

def split_periodic_step(p0, p1):
    """Split one unwrapped step at box boundaries into wrapped sub-segments.
    Sub-segments that still span more than half the box (boundary endpoints) are dropped.
    """
    d = p1 - p0
    ts = [0.0, 1.0]

    # Find t values where the unwrapped path crosses an x boundary
    if d[0] != 0.0:
        # All integer multiples of Lx between p0[0] and p1[0]
        lo, hi = (p0[0], p1[0]) if d[0] > 0 else (p1[0], p0[0])
        for n in range(int(np.floor(lo / Lx)), int(np.floor(hi / Lx)) + 2):
            xb = n * Lx
            tx = (xb - p0[0]) / d[0]
            if 0.0 < tx < 1.0:
                ts.append(float(tx))

    # Find t values where the unwrapped path crosses a y boundary
    if d[1] != 0.0:
        lo, hi = (p0[1], p1[1]) if d[1] > 0 else (p1[1], p0[1])
        for n in range(int(np.floor(lo / Ly)), int(np.floor(hi / Ly)) + 2):
            yb = n * Ly
            ty = (yb - p0[1]) / d[1]
            if 0.0 < ty < 1.0:
                ts.append(float(ty))

    ts = sorted(set(ts))
    segs = []

    for k in range(len(ts) - 1):
        t0, t1 = ts[k], ts[k + 1]
        a = p0 + t0 * d
        b = p0 + t1 * d
        aw = np.array([np.mod(a[0], Lx), np.mod(a[1], Ly)])
        bw = np.array([np.mod(b[0], Lx), np.mod(b[1], Ly)])

        # Skip segments whose wrapped endpoints still cross the box
        # (happens when one endpoint is exactly on the boundary: mod gives 0)
        if np.abs(bw[0] - aw[0]) > Lx / 2 or np.abs(bw[1] - aw[1]) > Ly / 2:
            continue
        if np.linalg.norm(bw - aw) < 1e-10:
            continue

        segs.append([aw, bw])

    return segs

def update_trail(frame_idx):
    global prev_wrapped, prev_unwrapped

    wrapped = np.genfromtxt(os.path.join(path, files[frame_idx]), delimiter=' ', skip_header=1)[:, :2]

    if frame_idx > 0:
        delta = wrapped - prev_wrapped
        delta[:, 0] = np.where(delta[:, 0] >  Lx/2, delta[:, 0] - Lx, delta[:, 0])
        delta[:, 0] = np.where(delta[:, 0] < -Lx/2, delta[:, 0] + Lx, delta[:, 0])
        delta[:, 1] = np.where(delta[:, 1] >  Ly/2, delta[:, 1] - Ly, delta[:, 1])
        delta[:, 1] = np.where(delta[:, 1] < -Ly/2, delta[:, 1] + Ly, delta[:, 1])
        unwrapped = prev_unwrapped + delta

        for i in range(n_particles):
            new_segs = split_periodic_step(prev_unwrapped[i], unwrapped[i])
            if new_segs:
                segment_histories[i].extend(new_segs)

            if segment_histories[i]:
                r, g, b, _ = particle_colors[i]
                n_segs = len(segment_histories[i])
                alphas_arr = np.linspace(0.08, 0.95, n_segs)
                rgba = np.column_stack([
                    np.full(n_segs, r),
                    np.full(n_segs, g),
                    np.full(n_segs, b),
                    alphas_arr
                ])
                line_collections[i].set_segments(segment_histories[i])
                line_collections[i].set_colors(rgba)

        prev_unwrapped = unwrapped

    prev_wrapped = wrapped
    scat_trail.set_offsets(wrapped)
    label_trail.set_text(f'frame: {frame_idx + 1}/{len(files)} | file: {files[frame_idx]}')
    return [scat_trail, label_trail, *line_collections]

ani_trail = FuncAnimation(
    fig_trail, update_trail, frames=len(files), interval=1000/fps, blit=False, repeat=False
)

output_trail_mp4 = 'vicsek_per_particle_colored.mp4'
writer_trail = FFMpegWriter(fps=fps, bitrate=2400)

try:
    ani_trail.save(output_trail_mp4, writer=writer_trail, dpi=160)
    print(f'Saved {output_trail_mp4}')
except FileNotFoundError:
    print('ffmpeg not found. Install ffmpeg, then run this cell again.')

plt.close(fig_trail)
